## Phase 2, Step 1: Training & Testing Checkpoints
**Goal:** Create specialized, permanent Parquet files for training and testing. This "Storage-Based" approach prevents kernel crashes and ensures the machine learning phase has maximum RAM headroom.

In [0]:
# 1. Install LightGBM using the magic command
%pip install lightgbm

# 2. Restart the Python interpreter to recognize the new library
# (Note: Databricks usually handles this automatically with %pip, but it's a good habit)

In [0]:
import pandas as pd
import gc

file_path = '/Volumes/workspace/default/m5_walmart_data/m5_silver_layer.parquet'

# 1. Load the data once
df = pd.read_parquet(file_path)

# 2. Derive day_int numerically to avoid string split crashes
if 'day_int' not in df.columns:
    df['day_int'] = df['day'].apply(lambda x: int(x[2:])).astype('int16')

# 3. Create the permanent TEST file (Last 28 days)
split_point = df['day_int'].max() - 28
test_df = df[df['day_int'] > split_point]
test_df.to_parquet('/Volumes/workspace/default/m5_walmart_data/m5_test.parquet')
del test_df
gc.collect()

# 4. Create the permanent TRAIN file (Sampled to ensure stability)
# Training on 5 million rows is enough for a high-quality portfolio result
train_df = df[df['day_int'] <= split_point].sample(n=5000000, random_state=42)
train_df.to_parquet('/Volumes/workspace/default/m5_walmart_data/m5_train_sample.parquet')

# 5. Clear everything
del df, train_df
gc.collect()

print("Checkpoints created! Now you can restart your kernel and proceed to training.")

## Phase 2, Step 2: Model Preparation & Feature Definition

In [0]:
import pandas as pd
import lightgbm as lgb
import gc

# Load the much smaller, pre-split files
train = pd.read_parquet('/Volumes/workspace/default/m5_walmart_data/m5_train_sample.parquet')
test = pd.read_parquet('/Volumes/workspace/default/m5_walmart_data/m5_test.parquet')

# Define Features
unused = ['id', 'sales', 'wm_yr_wk', 'day', 'day_int']
features = [c for c in train.columns if c not in unused]

X_train, y_train = train[features], train['sales']
X_test, y_test = test[features], test['sales']

del train, test
gc.collect()

print(f"Data ready for LightGBM. Training rows: {len(X_train)}")

## Phase 2, Step 3: Global LightGBM Training

In [0]:
import lightgbm as lgb
import gc

# 1. Define Categorical Features
# This is a "Senior" step: telling LightGBM which columns are categories 
# allows it to handle them more efficiently than One-Hot Encoding.
cat_features = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 
                'wday', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI']

# 2. Set "Senior-Level" Parameters
# We use a low learning rate and high number of estimators for precision.
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'sub_feature': 0.8,
    'sub_row': 0.75,
    'bagging_freq': 1,
    'lambda_l2': 0.1,
    'verbosity': 1,
    'num_iterations': 500, # Start with 500 to keep it fast
    'num_leaves': 128,
    'min_data_in_leaf': 50,
}

# 3. Initialize and Train
print("Starting Global LightGBM Training...")
model = lgb.LGBMRegressor(**params)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=cat_features,
    callbacks=[lgb.early_stopping(stopping_rounds=20)]
)

print("Model Training Complete.")

## Phase 2, Step 4: Model Evaluation (The "Proof")

In [0]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# 1. Generate Predictions
preds = model.predict(X_test)

# 2. Calculate Metrics
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

print(f"--- Model Evaluation ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

# 3. Quick Logic Check: Are we predicting negative sales?
if (preds < 0).any():
    print("Note: Model predicted some negative values. Clipping to zero...")
    preds = np.clip(preds, 0, None)

## Phase 2, Step 5: Model Serialization (The "Deployment" Piece)

In [0]:
import joblib
import shutil
import os

# 1. Define Paths
local_path = '/tmp/m5_forecasting_model.pkl'
volume_path = '/Volumes/workspace/default/m5_walmart_data/m5_forecasting_model.pkl'
native_local = '/tmp/m5_model_native.txt'
native_volume = '/Volumes/workspace/default/m5_walmart_data/m5_model_native.txt'

# 2. Save locally first (Standard local FS supports 'seek')
print("Serializing model to local /tmp storage...")
joblib.dump(model, local_path)
model.booster_.save_model(native_local)

# 3. Transfer from local to permanent Volume
print("Transferring artifacts to Databricks Volume...")
shutil.copyfile(local_path, volume_path)
shutil.copyfile(native_local, native_volume)

# 4. Clean up /tmp
os.remove(local_path)
os.remove(native_local)

print(f"Model successfully moved to: {volume_path}")

## Phase 2, Step 6: Feature Importance Analysis

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Extract feature importance
importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

# 2. Visualize the Top 10 signals
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importance_df.head(10))
plt.title('Top 10 Drivers of Walmart Sales Forecast')
plt.show()

print("Feature Importance Visualization Complete.")